# Supplementary videos S1–S8 ($D$ = 0.25, $f_{mut}$ = 0.25)

In [ ]:
import os
import gc
import dill
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
import pyprojroot.here as here
import vertex_lite as model
from vertex_lite import coloring, custom

In [ ]:
noise = 0.25
pct = 0.25
sim = 0

# Supplementary videos:
#   S1/S2: 'mut'       (random / cluster)  mutant vs wild-type
#   S3/S4: 'tef_color' (random / cluster)  effective edge tension
#   S5/S6: 'si'        (random / cluster)  shape index
#   S7/S8: 'nl'        (random / cluster)  number of neighbours
arrangements = ['random', 'cluster']
colorings = ['mut', 'tef_color', 'si', 'nl']

save_path = here('results/videos')
os.makedirs(save_path, exist_ok=True)

In [ ]:
def load_history(arrangement):
    folder_dir = here(f'results/resultados_{arrangement}')
    with open(os.path.join(folder_dir, f"noise_{noise}", f"pct_{pct}", f"sim_{sim}", "history.dill"), "rb") as f:
        return dill.load(f)


def add_colors(history):
    for cells in history:
        cells.properties['mut'] = coloring.definecolors_manual(cells, property_name='parent_group')
        cells.properties['shape_index'] = [custom.shape_index(cells, i, normalize=False, max_shape_index=5.0) for i in range(len(cells.mesh.area))]
        cells.properties['teff'] = [custom.calculate_t_eff(cells, i) for i in range(len(cells))]
        cells.properties['n_lados'] = np.array([len(cells.mesh.boundary(i)) for i in range(len(cells))])
        cells.properties['tef_color'] = coloring.values_to_hex_colors2(cells.properties['teff'], cmap_name='gnuplot', fixed_vmin=0.25, fixed_vmax=1.0)
        cells.properties['si'] = coloring.values_to_hex_colors2(cells.properties['shape_index'], cmap_name='custom', fixed_vmin=3.72, fixed_vmax=3.9)
        cells.properties['nl'] = coloring.definecolors_fixed_1_10_center6(cells, property_name='n_lados', cmap_name='BrBG')


def make_video(history, arrangement, tocolor):
    fig = plt.figure()
    fig.set_size_inches(6, 6)
    ax = fig.gca()

    v_max = np.max((np.max(history[-1].mesh.vertices), np.max(history[0].mesh.vertices)))
    size = 2.0 * v_max

    def init_fig():
        return (ax,)

    def animate_fig(i):
        cells = history[i]
        time = int(20 * i)  # one saved frame every 20 time units
        model.plotting.drawnoplt_videos(cells, ax, size, tocolor=tocolor)
        ax.set_title(f"Time: {time}")
        return (ax,)

    anim = animation.FuncAnimation(fig, animate_fig, frames=len(history), init_func=init_fig)
    filename = f'{save_path}/video_{arrangement}_{tocolor}_pct_{pct}_noise_{noise}.mp4'
    anim.save(filename, fps=30, extra_args=['-vcodec', 'libx264'], writer='ffmpeg')
    plt.close(fig)
    print(f"Saved: {filename}")

In [ ]:
for arrangement in arrangements:
    history = load_history(arrangement)
    add_colors(history)
    for tocolor in colorings:
        make_video(history, arrangement, tocolor)
    del history
    gc.collect()